# Replication: Lin et al. NN Matching Experiments + Local-Polynomial Extension (genriesz)

This notebook is a **self-contained Python replication harness** for the experimental setup in:

- Z. Lin, P. Ding, and F. Han (Econometrica, forthcoming), *Estimation based on nearest neighbor matching: from density ratio to average treatment effect*.

It also adds an **experimental local-polynomial variant** of NN matching via **local-polynomial NN-LSIF** (least-squares density-ratio estimation) implemented in `genriesz`.

## Data / replication package

Please download the Lin et al. replication package + data from:

- https://zenodo.org/records/8322609

After unzipping, you should have a folder that contains a `data/` subfolder.  
This notebook expects at least:

- `data/exp_generated.feather` (simulated LaLonde / LaLonde-style data)

> **Note:** Reading Feather files requires `pyarrow`. If you do not have it installed, run:
>
> ```bash
> pip install pyarrow
> ```


In [ ]:
import os
from pathlib import Path

import numpy as np

# Pandas is used for convenience; Feather reading requires pyarrow.
try:
    import pandas as pd
except Exception as e:
    raise ImportError("This notebook requires pandas. Please install pandas.") from e

# Import genriesz.
try:
    import genriesz
except ImportError:
    # If running from the source tree (this repo), add ../src to sys.path.
    import sys
    sys.path.insert(0, str(Path("..").resolve() / "src"))
    import genriesz

print("genriesz version:", getattr(genriesz, "__version__", "unknown"))


In [ ]:
# ---------------------------------------------------------------------
# Point this to the *unzipped* Lin et al. replication folder.
#
# The folder must contain a `data/` subfolder. For example:
#   LIN_ROOT = Path("/path/to/lin_replication_package")
# ---------------------------------------------------------------------

LIN_ROOT = Path("lin_replication_package")  # <-- CHANGE ME

DATA_DIR = LIN_ROOT / "data"
print("LIN_ROOT:", LIN_ROOT.resolve())
print("DATA_DIR exists?", DATA_DIR.exists())


In [ ]:
# Load the simulated LaLonde data used by Lin et al. (exp_generated.feather).

feather_path = DATA_DIR / "exp_generated.feather"
if not feather_path.exists():
    raise FileNotFoundError(
        f"Could not find {feather_path}.\n"
        "Please download the replication package from Zenodo (link above), unzip it, "
        "and set LIN_ROOT accordingly."
    )

try:
    df = pd.read_feather(feather_path)  # requires pyarrow
except Exception as e:
    raise RuntimeError(
        "Failed to read the Feather file. Make sure `pyarrow` is installed:\n"
        "    pip install pyarrow"
    ) from e

print("Loaded:", feather_path)
print("Shape:", df.shape)
df.head()


In [ ]:
# Covariates used in Lin et al. (see comparison.R).
COVARIATES = [
    "black", "hispanic", "married", "nodegree",
    "re74", "re75", "education", "age",
]

# Treatment and outcome columns in Lin's dataset.
T_COL = "t"
Y_COL = "re78"

# Optional counterfactual outcome column (used to compute the Monte Carlo 'true ATE').
Y_CF_COL = "re78_cf"

missing = [c for c in COVARIATES + [T_COL, Y_COL] if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

X_cov = df[COVARIATES].to_numpy(dtype=float)
D = df[T_COL].to_numpy(dtype=int)
Y = df[Y_COL].to_numpy(dtype=float)

print("X_cov:", X_cov.shape, "D:", D.shape, "Y:", Y.shape)
print("Pr(D=1):", D.mean())


In [ ]:
# Lin et al. compute a "true ATE" from the simulated counterfactual outcomes.
# This matches the computation in produce_tables.R.

if Y_CF_COL in df.columns:
    Y_cf = df[Y_CF_COL].to_numpy(dtype=float)
    te = ((D == 1).astype(float) - (D == 0).astype(float)) * (Y - Y_cf)

    # In Lin's code, the treated/control sampling proportions are 185 and 260.
    p1 = 185.0 / (185.0 + 260.0)
    p0 = 260.0 / (185.0 + 260.0)
    ate_true = p1 * te[D == 1].mean() + p0 * te[D == 0].mean()
    print("True ATE (Lin et al. scaling):", ate_true)
else:
    ate_true = None
    print("No counterfactual outcome column found; ate_true will be None.")


## Estimator implementations

In [ ]:
from scipy.spatial import cKDTree

def standardize_columns(X: np.ndarray):
    """Match R's scale(X): mean 0, std 1 column-wise."""
    X = np.asarray(X, dtype=float)
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd[sd == 0.0] = 1.0
    return (X - mu) / sd, mu, sd

def poly2_per_feature_design(X: np.ndarray):
    """A simple 'series' design: [1, x1, x1^2, x2, x2^2, ...]."""
    X = np.asarray(X, dtype=float)
    n, d = X.shape
    out = np.empty((n, 1 + 2 * d), dtype=float)
    out[:, 0] = 1.0
    out[:, 1::2] = X
    out[:, 2::2] = X ** 2
    return out

def fit_series_poly2(Y: np.ndarray, X: np.ndarray):
    """Fit least squares on the per-feature quadratic series design."""
    Phi = poly2_per_feature_design(X)
    beta, *_ = np.linalg.lstsq(Phi, Y, rcond=None)
    return beta

def predict_series_poly2(beta: np.ndarray, X: np.ndarray) -> np.ndarray:
    Phi = poly2_per_feature_design(X)
    return Phi @ beta

def knn_indices(X_train: np.ndarray, X_query: np.ndarray, k: int, leafsize: int = 16) -> np.ndarray:
    """kNN indices in X_train for each row of X_query."""
    tree = cKDTree(X_train, leafsize=int(leafsize))
    _dist, idx = tree.query(X_query, k=int(k))
    idx = np.asarray(idx)
    if idx.ndim == 1:
        idx = idx.reshape(-1, 1)
    return idx


In [ ]:
def bcm_estimate(
    X: np.ndarray,
    Y: np.ndarray,
    D: np.ndarray,
    *,
    M: int,
    leafsize: int = 16,
):
    """Bias-corrected matching (BCM) as in Lin et al.'s matching.R.

    This mirrors the R function `mbc(X,Y,Tr,M,Model1,Model0)` with Model1/Model0
    coming from the simple polynomial series regression.

    Returns
    -------
    est, se, AIse
    """
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float).reshape(-1)
    D = np.asarray(D, dtype=int).reshape(-1)

    if X.ndim != 2:
        raise ValueError("X must be 2D.")
    n = len(X)
    if len(Y) != n or len(D) != n:
        raise ValueError("X, Y, D must have the same length.")

    # Standardize covariates (Lin et al. use scale(X)).
    Xs, _mu, _sd = standardize_columns(X)

    X1 = Xs[D == 1]
    X0 = Xs[D == 0]
    Y1 = Y[D == 1]
    Y0 = Y[D == 0]

    n1 = len(X1)
    n0 = len(X0)
    if n1 == 0 or n0 == 0:
        raise ValueError("Both treatment arms must be nonempty.")
    if M > n1 or M > n0:
        raise ValueError(f"M={M} is too large for group sizes (n1={n1}, n0={n0}).")

    # Outcome models (series regression) fit on each arm, predict on full sample.
    b1 = fit_series_poly2(Y1, X1)
    b0 = fit_series_poly2(Y0, X0)
    Model1 = predict_series_poly2(b1, Xs)
    Model0 = predict_series_poly2(b0, Xs)

    # Cross-group nearest neighbors.
    # Index1: for each control, M nearest treated indices (0..n1-1).
    Index1 = knn_indices(X1, X0, k=M, leafsize=leafsize)
    # Index0: for each treated, M nearest control indices (0..n0-1).
    Index0 = knn_indices(X0, X1, k=M, leafsize=leafsize)

    # Matched-times counts (normalized by M).
    K1M = np.bincount(Index1.reshape(-1), minlength=n1).astype(float) / float(M)
    K0M = np.bincount(Index0.reshape(-1), minlength=n0).astype(float) / float(M)

    # Residual terms (Lin's notation).
    Res1 = (1.0 + K1M) * (Y[D == 1] - Model1[D == 1])
    Res0 = (1.0 + K0M) * (Y[D == 0] - Model0[D == 0])

    est = float(np.mean(Model1 - Model0) + np.mean(np.concatenate([Res1, -Res0])))

    # Naive (Wald-style) standard error used in Lin's code.
    diff = (Model1 - Model0) - est
    psi1 = diff[D == 1] + Res1
    psi0 = diff[D == 0] - Res0
    se = float(np.sqrt(np.mean(np.concatenate([psi1 ** 2, psi0 ** 2])) / n))

    # Abadie–Imbens-type variance estimator (as in Lin's code).
    Y1hat = Y1[Index1].mean(axis=1)  # impute for controls
    Y0hat = Y0[Index0].mean(axis=1)  # impute for treated
    AIvar1 = float(np.mean(np.concatenate([(Y1hat - Y0 - est) ** 2, (Y1 - Y0hat - est) ** 2])) / n)

    # Within-group 1-NN indices (exclude self by using k=2 and taking the 2nd).
    tree1 = cKDTree(X1, leafsize=int(leafsize))
    _d1, idx1 = tree1.query(X1, k=2)
    nn1 = idx1[:, 1]
    tree0 = cKDTree(X0, leafsize=int(leafsize))
    _d0, idx0 = tree0.query(X0, k=2)
    nn0 = idx0[:, 1]

    varhat1 = (Y1 - Y1[nn1]) ** 2 / 2.0
    varhat0 = (Y0 - Y0[nn0]) ** 2 / 2.0

    AIvar2 = float(
        np.mean(
            np.concatenate(
                [
                    (K1M ** 2 + (2.0 - 1.0 / float(M)) * K1M) * varhat1,
                    (K0M ** 2 + (2.0 - 1.0 / float(M)) * K0M) * varhat0,
                ]
            )
        )
        / n
    )

    AIse = float(np.sqrt(AIvar1 + AIvar2))
    return est, se, AIse


In [ ]:
def bcm_local_polynomial_weights(
    X: np.ndarray,
    D: np.ndarray,
    *,
    M: int,
    degree: int,
    kernel: str = "knn_ball",
):
    """Compute local-polynomial NN-LSIF weights via genriesz.

    Parameters
    ----------
    kernel:
        Either ``'knn_ball'`` or ``'catchment'``.

        * ``'knn_ball'``: kNN-ball localization (default).
        * ``'catchment'``: catchment-area localization.
    """
    X = np.asarray(X, dtype=float)
    D = np.asarray(D, dtype=int).reshape(-1)
    X_full = np.column_stack([D, X])

    w = genriesz.local_polynomial_nnlsif_weights(
        X=X_full,
        treatment_index=0,
        M=int(M),
        degree=int(degree),
        kernel=str(kernel),
        standardize=True,  # match Lin's scale(X)
    )
    # Return group-specific weights.
    w1 = w[D == 1]
    w0 = w[D == 0]
    return w1, w0


def bcm_estimate_local_polynomial(
    X: np.ndarray,
    Y: np.ndarray,
    D: np.ndarray,
    *,
    M: int,
    degree: int,
    kernel: str = "knn_ball",
    leafsize: int = 16,
):
    """A BCM-style estimator using local-polynomial NN-LSIF weights.

    This mirrors the ARW structure:
        mean(Model1 - Model0) + mean( w1*(Y-Model1) ) - mean( w0*(Y-Model0) )

    Standard errors are computed using the same plug-in influence-function style
    formula as in Lin et al., but **AIse is not implemented** for this variant.
    """
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float).reshape(-1)
    D = np.asarray(D, dtype=int).reshape(-1)

    # Standardize covariates for the outcome series fit to match Lin's setup.
    Xs, _mu, _sd = standardize_columns(X)

    X1 = Xs[D == 1]
    X0 = Xs[D == 0]
    Y1 = Y[D == 1]
    Y0 = Y[D == 0]

    # Outcome models.
    b1 = fit_series_poly2(Y1, X1)
    b0 = fit_series_poly2(Y0, X0)
    Model1 = predict_series_poly2(b1, Xs)
    Model0 = predict_series_poly2(b0, Xs)

    # Local-polynomial weights (computed from the original covariates; genriesz standardizes internally).
    w1, w0 = bcm_local_polynomial_weights(X, D, M=M, degree=degree, kernel=kernel)

    Res1 = w1 * (Y[D == 1] - Model1[D == 1])
    Res0 = w0 * (Y[D == 0] - Model0[D == 0])

    est = float(np.mean(Model1 - Model0) + np.mean(np.concatenate([Res1, -Res0])))

    n = len(X)
    diff = (Model1 - Model0) - est
    psi1 = diff[D == 1] + Res1
    psi0 = diff[D == 0] - Res0
    se = float(np.sqrt(np.mean(np.concatenate([psi1 ** 2, psi0 ** 2])) / n))

    AIse = np.nan
    return est, se, AIse



## Quick sanity check

In [ ]:
# Quick single-run sanity check (on a single sampled dataset).
rng = np.random.default_rng(123)

N = 1200
p_treated = 185.0 / (185.0 + 260.0)
N_treated = int(np.floor(N * p_treated))
N_control = int(N - N_treated)

treated_rows = np.flatnonzero(D == 1)
control_rows = np.flatnonzero(D == 0)

idx = np.concatenate(
    [
        rng.choice(treated_rows, size=N_treated, replace=False),
        rng.choice(control_rows, size=N_control, replace=False),
    ]
)
X_s = X_cov[idx]
Y_s = Y[idx]
D_s = D[idx]

est, se, aise = bcm_estimate(X_s, Y_s, D_s, M=4)
print("BCM (Lin) est/se/AIse:", est, se, aise)

# Local polynomial variant (degree=1): compare kernels.
for kernel in ["knn_ball", "catchment"]:
    est_lp, se_lp, _ = bcm_estimate_local_polynomial(X_s, Y_s, D_s, M=4, degree=1, kernel=kernel)
    print(f"BCM + local-poly weights (degree=1, kernel={kernel}) est/se:", est_lp, se_lp)


## Monte Carlo loop (Lin et al. settings)

Lin et al. run a fairly heavy Monte Carlo study (e.g. 2000 repetitions for each `(N, M)` setting).
This can take many hours.

In the cell below, **reduce `N_RUNS` first** to validate the pipeline, then increase it if needed.


In [ ]:
# Lin et al. settings (see comparison.R).
N_LIST = [600, 1200, 4800, 9600]
M_FIX_LIST = [1, 4, 16]
ALPHA_LIST = [0.5, 1, 2, 5, 10]

# Adjust this for runtime.
N_RUNS = 50   # Lin et al. use 2000
SEED = 123

# Local polynomial degrees to evaluate (set to [] to skip).
LP_DEGREES = [1]   # e.g., [1, 2]; degree=0 corresponds to local-constant

# Compare both canonical localization kernels.
LP_KERNELS = ["knn_ball", "catchment"]

rng = np.random.default_rng(SEED)

treated_rows = np.flatnonzero(D == 1)
control_rows = np.flatnonzero(D == 0)

records = []

for N in N_LIST:
    p_treated = 185.0 / (185.0 + 260.0)
    N_treated = int(np.floor(N * p_treated))
    N_control = int(N - N_treated)

    M_list = list(M_FIX_LIST) + [int(np.floor(a * (N ** (1.0 / 3.0)))) for a in ALPHA_LIST]
    M_list = sorted(set(M_list))

    for M in M_list:
        if M <= 0:
            continue
        if M > N_treated or M > N_control:
            continue

        for r in range(N_RUNS):
            idx = np.concatenate(
                [
                    rng.choice(treated_rows, size=N_treated, replace=False),
                    rng.choice(control_rows, size=N_control, replace=False),
                ]
            )
            X_s = X_cov[idx]
            Y_s = Y[idx]
            D_s = D[idx]

            # --- Lin's BCM (matching weights)
            est, se, aise = bcm_estimate(X_s, Y_s, D_s, M=M)
            records.append(
                dict(
                    method="BCM_ps",
                    variant="matching",
                    N=N,
                    M=M,
                    run=r,
                    est=est,
                    se=se,
                    AIse=aise,
                )
            )

            # --- Local polynomial extension(s)
            for deg in LP_DEGREES:
                for kernel in LP_KERNELS:
                    est_lp, se_lp, _ = bcm_estimate_local_polynomial(
                        X_s,
                        Y_s,
                        D_s,
                        M=M,
                        degree=deg,
                        kernel=kernel,
                    )
                    records.append(
                        dict(
                            method="BCM_ps",
                            variant=f"local_poly_{kernel}_deg{deg}",
                            N=N,
                            M=M,
                            run=r,
                            est=est_lp,
                            se=se_lp,
                            AIse=np.nan,
                        )
                    )

print("Completed records:", len(records))
runs_df = pd.DataFrame.from_records(records)
runs_df.head()


## Summary table (RMSE / bias / SD / coverage)

In [ ]:
# Summarize results similarly to Lin et al.'s produce_tables.R.

if ate_true is None:
    raise RuntimeError("ate_true is None. The dataset must include re78_cf to compute it.")

out = runs_df.copy()
out["error"] = (out["est"] - ate_true) / 1000.0
out["se"] = out["se"] / 1000.0
out["AIse"] = out["AIse"] / 1000.0

out["covered95"] = np.abs(out["error"]) < 1.96 * out["se"]
out["AIcovered95"] = np.where(np.isfinite(out["AIse"]), np.abs(out["error"]) < 1.96 * out["AIse"], np.nan)

summary = (
    out.groupby(["variant", "N", "M"], as_index=False)
      .agg(
          n_runs=("error", "size"),
          rmse=("error", lambda x: float(np.sqrt(np.mean(x**2)))),
          bias=("error", lambda x: float(np.mean(x))),
          sd=("error", lambda x: float(np.sqrt(np.mean((x - np.mean(x))**2)))),
          mae=("error", lambda x: float(np.mean(np.abs(x)))),
          coverage95=("covered95", "mean"),
          AIcoverage95=("AIcovered95", "mean"),
      )
      .sort_values(["variant", "N", "M"])
)

summary
